In [1]:
!pip install langchain langchain-chroma langchain-google-genai pymongo python-dotenv

In [2]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 1. Load Environment Variables
load_dotenv("../.env")

# 2. Connect to MongoDB
MONGODB_URI = os.getenv("mongodb")
client = MongoClient(MONGODB_URI)

# Using the 'local' db and 'pdf_summaries' collection used in the previous steps
db = client.local
collection = db.pdf_summaries

mongo_docs = list(collection.find({}))
print(f"Fetched {len(mongo_docs)} documents from MongoDB.")

c:\Users\vaibh\OneDrive\Desktop\ML\Projects\ecommendation-system\Policy-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OperationFailure: (Unauthorized) not authorized on local to execute command { find: "pdf_summaries", filter: {  }, lsid: { id: {4 [23 114 85 82 103 81 73 193 131 206 194 190 241 202 210 31]} }, $clusterTime: { clusterTime: {1775484486 6}, signature: { hash: {0 [77 249 217 176 63 236 228 107 67 48 76 70 124 236 49 98 194 149 41 121]}, keyId: 7592286076518006784.000000 } }, $db: "local" }, full error: {'ok': 0, 'errmsg': '(Unauthorized) not authorized on local to execute command { find: "pdf_summaries", filter: {  }, lsid: { id: {4 [23 114 85 82 103 81 73 193 131 206 194 190 241 202 210 31]} }, $clusterTime: { clusterTime: {1775484486 6}, signature: { hash: {0 [77 249 217 176 63 236 228 107 67 48 76 70 124 236 49 98 194 149 41 121]}, keyId: 7592286076518006784.000000 } }, $db: "local" }', 'code': 8000, 'codeName': 'AtlasError'}

In [ ]:
# 3. Create Langchain Document Objects
langchain_docs = []
for doc in mongo_docs:
    text_content = doc.get("summary", "")
    
    if text_content:
        # Store useful reference data in the metadata
        metadata = {
            "source_path": doc.get("path", ""),
            "mongo_id": str(doc.get("_id", ""))
        }
        
        lc_doc = Document(page_content=text_content, metadata=metadata)
        langchain_docs.append(lc_doc)
        
print(f"Created {len(langchain_docs)} Langchain Document objects.")

In [ ]:
# 4. Generate Embeddings & Store in Vector DB (Chroma)
import warnings
warnings.filterwarnings('ignore') # Ignore grpc warnings if any

# Initialize embedding model (from previous Google Config)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

print("Generating embeddings... This might take a moment.")

# Create and persist Chroma vector store
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=embeddings,
    persist_directory="./chroma_db_summaries" # Saves to local disk
)

print("\u2705 Embeddings successfully generated and persistent in './chroma_db_summaries'!")

In [ ]:
# 5. Test Vector Similarity Search
query = "What does this policy say about wastewater or water treatment?"
results = vectorstore.similarity_search(query, k=2)

print(f"Found {len(results)} relevant results:\n")
for i, match in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {match.metadata.get('source_path')}")
    print(f"Snippet: {match.page_content[:250]}...\n")

In [ ]:
from langchain_groq import ChatGroq

# 1. Initialize Groq model
groq_model = ChatGroq(model="llama-3.3-70b-versatile")
query = "What does this policy say about wastewater or water treatment?"

# 2. Request a hypothetical short answer to the query
prompt = f"Write a short, factual, and informative paragraph answering this query, pretending you are a factual policy document:\n\nQuery: {query}\n\nAnswer:"
hypothetical_answer_response = groq_model.invoke(prompt)
hypothetical_answer = hypothetical_answer_response.content

print("========== HYPOTHETICAL ANSWER (From Groq) ==========")
print(hypothetical_answer)
print("=====================================================\n")

# 3. Run search using the hypothetical answer as the query text 
# (similarity_search automatically generates the embedding for the new answer string)
advanced_results = vectorstore.similarity_search(hypothetical_answer, k=2)

print(f"Found {len(advanced_results)} relevant results based on Groq's generated contextual answer:\n")
for i, match in enumerate(advanced_results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {match.metadata.get('source_path')}")
    print(f"Snippet: {match.page_content[:300]}...\n")
